# 05 Power BI Dashboard Preparation

## Project: PG&E-Style Utility Operations & Meter-to-Cash Analytics

This notebook prepares the Power BI dashboard design for the project. The goal is to define the reporting datasets, dashboard pages, data model relationships, KPIs, visuals, and business questions that the final dashboard should answer.

The earlier notebooks created the real-data foundation, synthetic Meter-to-Cash data model, SQL database layer, SQL analysis views, and geospatial outage mapping outputs. This notebook translates those outputs into a dashboard plan.

In this notebook, I will:

1. Identify the reporting-ready files to load into Power BI.
2. Define the dashboard pages and business purpose of each page.
3. Document the Power BI data model and relationships.
4. Define key measures and KPIs.
5. Plan visuals for executive, billing, exception, service request, account readiness, and outage context pages.
6. Document dashboard design decisions for the final README and project presentation.

The purpose of this notebook is to make the Power BI build structured, business-focused, and aligned with the project’s Meter-to-Cash and utility operations goals.

## 1. Reporting Files for Power BI

This section lists the reporting-ready files that will be loaded into Power BI. Most dashboard visuals will use the SQL view exports from `data/reporting`, while the geospatial page will use outputs from `data/geospatial`.

In [1]:
import pandas as pd
from pathlib import Path

# Define paths
REPORTING_DIR = Path("../data/reporting")
GEOSPATIAL_DIR = Path("../data/geospatial")
SYNTHETIC_DIR = Path("../data/synthetic")
PROCESSED_DIR = Path("../data/processed")

# List reporting and geospatial files
reporting_files = sorted([file.name for file in REPORTING_DIR.iterdir()])
geospatial_files = sorted([file.name for file in GEOSPATIAL_DIR.iterdir()])

print("Reporting files:")
display(reporting_files)

print("Geospatial files:")
display(geospatial_files)

Reporting files:


['vw_account_readiness.csv',
 'vw_bill_status_summary.csv',
 'vw_billing_exception_summary.csv',
 'vw_meter_to_cash_monthly_kpis.csv',
 'vw_monthly_billing_trend.csv',
 'vw_pge_outage_consumption_context.csv',
 'vw_service_request_backlog.csv']

Geospatial files:


['pge_geojson_vs_processed_county_comparison.csv',
 'pge_outage_area_county_summary.csv',
 'pge_outage_area_properties_clean.csv',
 'pge_outage_areas_clean.geojson']

### Power BI Input File Inventory

Primary reporting files:

- `vw_meter_to_cash_monthly_kpis.csv`
- `vw_monthly_billing_trend.csv`
- `vw_bill_status_summary.csv`
- `vw_billing_exception_summary.csv`
- `vw_service_request_backlog.csv`
- `vw_account_readiness.csv`
- `vw_pge_outage_consumption_context.csv`

Geospatial and outage mapping files:

- `pge_outage_area_county_summary.csv`
- `pge_outage_area_properties_clean.csv`
- `pge_outage_areas_clean.geojson`

The SQL view exports from `data/reporting` will serve as the primary Power BI data sources because they are already aggregated and reporting-ready. The geospatial outputs from `data/geospatial` will support outage mapping and county-level outage context.

## 2. Dashboard Page Design

This section defines the planned Power BI dashboard pages. Each page is designed around a specific business purpose and uses the reporting-ready SQL view outputs created in Notebook 3.

### Planned Dashboard Pages

| Page | Purpose | Primary Data Sources |
|---|---|---|
| Executive Overview | Provide high-level Meter-to-Cash KPIs, billing volume, bill amount, usage, exceptions, and service request activity. | `vw_meter_to_cash_monthly_kpis`, `vw_monthly_billing_trend` |
| Billing Performance | Analyze bill status, bill amount, usage, past-due bills, and customer segment performance. | `vw_bill_status_summary`, `vw_monthly_billing_trend` |
| Billing Exceptions | Monitor exception volume, severity, resolution status, exception type, and affected accounts. | `vw_billing_exception_summary` |
| Service Request Backlog | Track operational follow-up work by request type, priority, status, county, and customer segment. | `vw_service_request_backlog` |
| Account Readiness | Identify accounts blocked by missing reads, invalid reads, inactive status, open exceptions, or open service requests. | `vw_account_readiness` |
| PG&E Outage Context | Show current PG&E outage impact by county and connect outage activity to electricity consumption context. | `vw_pge_outage_consumption_context`, `pge_outage_area_county_summary`, `pge_outage_area_properties_clean` |

This page structure separates executive reporting, billing operations, exception management, service request workflows, data quality readiness, and real PG&E outage context.

## 3. Power BI Data Model Design

This section defines how the reporting tables should be loaded and related in Power BI. Because the SQL views were already created for reporting, the Power BI model can remain simple and page-focused.

### Recommended Power BI Tables to Load

The reporting CSV files created in Notebook 3 are the source outputs, but Power BI should load the cleaned Excel workbook created later in this notebook:

`data/powerbi/pge_powerbi_tables.xlsx`

This workbook contains one sheet per reporting table and avoids CSV parsing issues caused by regional decimal and date settings.

| Power BI Table | Workbook Sheet | Original Source File | Use |
|---|---|---|---|
| `Monthly KPIs` | `Monthly KPIs` | `vw_meter_to_cash_monthly_kpis.csv` | Executive KPI cards and monthly operational overview. |
| `Monthly Billing Trend` | `Monthly Billing Trend` | `vw_monthly_billing_trend.csv` | Monthly bill amount, usage, bill count, exception rate, and past-due trend visuals. |
| `Bill Status Summary` | `Bill Status Summary` | `vw_bill_status_summary.csv` | Bill status analysis by cycle, customer segment, county, rate plan, and status. |
| `Billing Exception Summary` | `Billing Exceptions` | `vw_billing_exception_summary.csv` | Exception analysis by type, severity, resolution status, segment, county, and billing cycle. |
| `Service Request Backlog` | `Service Requests` | `vw_service_request_backlog.csv` | Service request workflow analysis by request type, priority, status, segment, county, and billing cycle. |
| `Account Readiness` | `Account Readiness` | `vw_account_readiness.csv` | Account-level readiness, data quality blockers, and migration-readiness analysis. |
| `Outage Consumption Context` | `Outage Context` | `vw_pge_outage_consumption_context.csv` | PG&E county outage impact and 2024 electricity consumption context. |
| `Outage Area County Summary` | `Outage Area County` | `pge_outage_area_county_summary.csv` | County-level outage area map/table support. |
| `Outage Area Properties` | `Outage Area Detail` | `pge_outage_area_properties_clean.csv` | Outage area detail table and potential map tooltip support. |

### Recommended Relationships

Because the reporting files are already aggregated SQL outputs, the Power BI model should avoid unnecessary relationships that could create duplicate counting.

Recommended relationships:

| From Table | Field | To Table | Field | Relationship Type | Notes |
|---|---|---|---|---|---|
| `Monthly KPIs` | `billing_cycle_id` | `Monthly Billing Trend` | `billing_cycle_id` | Many-to-one or inactive | Use carefully; these tables can also be used independently. |
| `Bill Status Summary` | `billing_cycle_id` | `Monthly Billing Trend` | `billing_cycle_id` | Many-to-one | Useful for shared month filtering. |
| `Billing Exception Summary` | `billing_cycle_id` | `Monthly Billing Trend` | `billing_cycle_id` | Many-to-one | Useful for filtering exceptions by month. |
| `Service Request Backlog` | `billing_cycle_id` | `Monthly Billing Trend` | `billing_cycle_id` | Many-to-one | Useful for filtering service requests by month. |
| `Outage Area County Summary` | `county` | `Outage Consumption Context` | `county` | Many-to-one | Useful for combining polygon/count area context with county outage/consumption context. |
| `Outage Area Properties` | `county` | `Outage Consumption Context` | `county` | Many-to-one | Useful for county filters and tooltips. |

Design note:

For the first dashboard version, it will be safer to keep most reporting tables independent and use slicers from the same table on each page. This avoids accidental many-to-many filtering issues. If a shared date or county dimension is needed later, I will create small dimension tables for `billing_cycle_id` and `county`.

## 4. Dashboard KPIs and Measures

This section defines the main KPIs and Power BI measures for the dashboard. These measures will support executive reporting, billing performance analysis, exception monitoring, service request backlog tracking, account readiness, and outage context reporting.

### Core Executive KPIs

| KPI | Business Meaning | Recommended Source |
|---|---|---|
| Total Bills | Total number of bills generated across selected period. | `Monthly KPIs` or `Monthly Billing Trend` |
| Total Bill Amount | Total calculated bill amount for non-exception bills. | `Monthly KPIs` or `Monthly Billing Trend` |
| Total kWh Usage | Total billed usage across selected period. | `Monthly KPIs` or `Monthly Billing Trend` |
| Exception Bills | Number of bills blocked by exception logic. | `Monthly KPIs` |
| Bill Exception Rate | Share of bills in exception status. | `Monthly KPIs` |
| Past Due Rate | Share of bills in past-due status. | `Monthly KPIs` |
| Open Billing Exceptions | Number of unresolved billing exceptions. | `Monthly KPIs` or `Billing Exception Summary` |
| Open Service Requests | Number of open operational follow-up requests. | `Monthly KPIs` or `Service Request Backlog` |
| Ready Accounts | Number or percentage of accounts marked ready. | `Account Readiness` |
| Impacted Customers | Current PG&E outage impacted customers by county. | `Outage Consumption Context` |

### Recommended DAX Measures

The following DAX measures can be created in Power BI after loading the cleaned Power BI workbook.

```DAX
Total Bills = SUM('Monthly Billing Trend'[total_bills])

Total Bill Amount = SUM('Monthly Billing Trend'[total_bill_amount])

Average Bill Amount = AVERAGE('Monthly Billing Trend'[avg_bill_amount])

Total kWh Usage = SUM('Monthly Billing Trend'[total_kwh_usage])

Paid Bills = SUM('Monthly Billing Trend'[paid_bills])

Generated Bills = SUM('Monthly Billing Trend'[generated_bills])

Past Due Bills = SUM('Monthly Billing Trend'[past_due_bills])

Exception Bills = SUM('Monthly Billing Trend'[exception_bills])

Bill Exception Rate % =
DIVIDE(
    [Exception Bills],
    [Total Bills],
    0
)

Past Due Rate % =
DIVIDE(
    [Past Due Bills],
    [Total Bills],
    0
)

Open Exceptions =
SUM('Billing Exception Summary'[open_exceptions])

In Review Exceptions =
SUM('Billing Exception Summary'[in_review_exceptions])

Resolved Exceptions =
SUM('Billing Exception Summary'[resolved_exceptions])

High Severity Exceptions =
SUM('Billing Exception Summary'[high_severity_exceptions])

Open Service Requests =
SUM('Service Request Backlog'[open_requests])

In Progress Service Requests =
SUM('Service Request Backlog'[in_progress_requests])

Closed Service Requests =
SUM('Service Request Backlog'[closed_requests])

Ready Accounts =
CALCULATE(
    COUNTROWS('Account Readiness'),
    'Account Readiness'[readiness_flag] = 1
)

Not Ready Accounts =
CALCULATE(
    COUNTROWS('Account Readiness'),
    'Account Readiness'[readiness_flag] = 0
)

Account Readiness Rate % =
DIVIDE(
    [Ready Accounts],
    [Ready Accounts] + [Not Ready Accounts],
    0
)

Total Impacted Customers =
SUM('Outage Consumption Context'[total_impacted_customers])

Total Outage Incidents =
SUM('Outage Consumption Context'[outage_incidents])

Avg Impacted Customers per Annual GWh =
AVERAGE('Outage Consumption Context'[impacted_customers_per_annual_gwh])

### Measure Design Notes

These measures are intentionally simple because most aggregation logic was already handled in the SQL reporting views.

Key design choices:

- Billing trend measures should primarily use `Monthly Billing Trend`.
- Exception measures should primarily use `Billing Exception Summary`.
- Service request measures should primarily use `Service Request Backlog`.
- Readiness measures should use `Account Readiness`.
- Outage context measures should use `Outage Consumption Context`.
- Percent measures should use `DIVIDE()` to avoid divide-by-zero errors.

This keeps the Power BI model easier to maintain and reduces the risk of double counting.

## 5. Dashboard Page Specifications

This section defines each Power BI dashboard page in more detail, including the business question, recommended visuals, slicers, and primary data sources.

### Page 1: Executive Overview

**Business question:**  
How is the overall Meter-to-Cash process performing across billing, usage, exceptions, service requests, and outage context?

**Primary data sources:**

- `Monthly KPIs`
- `Monthly Billing Trend`
- `Outage Consumption Context`

**KPI cards:**

- Total Bills
- Total Bill Amount
- Total kWh Usage
- Bill Exception Rate %
- Past Due Rate %
- Open Billing Exceptions
- Open Service Requests
- Account Readiness Rate %

**Visuals:**

| Visual | Fields |
|---|---|
| KPI cards | Total Bills, Total Bill Amount, Total kWh Usage, Exception Rate, Past Due Rate |
| Line chart | `billing_cycle_id` vs `total_bill_amount` |
| Line chart | `billing_cycle_id` vs `total_kwh_usage` |
| Clustered column chart | `billing_cycle_id` by paid, generated, past-due, and exception bills |
| Bar chart | Top counties by impacted customers |
| Table | Monthly KPI summary |

**Slicers:**

- Billing Cycle
- Customer Segment, if using linked detail pages
- County, if using outage context

### Page 2: Billing Performance

**Business question:**  
How are bills performing by status, customer segment, county, rate plan, and month?

**Primary data sources:**

- `Bill Status Summary`
- `Monthly Billing Trend`

**KPI cards:**

- Total Bills
- Total Bill Amount
- Average Bill Amount
- Paid Bills
- Past Due Bills
- Exception Bills

**Visuals:**

| Visual | Fields |
|---|---|
| Stacked column chart | `billing_cycle_id` by `bill_status` and `bill_count` |
| Bar chart | `customer_segment` by `total_bill_amount` |
| Bar chart | `county` by `total_bill_amount` |
| Matrix | `customer_segment`, `bill_status`, `bill_count`, `total_bill_amount`, `avg_bill_amount` |
| Line chart | `billing_cycle_id` vs `avg_bill_amount` |
| Scatter plot | `avg_kwh_usage` vs `avg_bill_amount` by customer segment or county |

**Slicers:**

- Billing Cycle
- Bill Status
- Customer Segment
- County
- Rate Plan

### Page 3: Billing Exceptions

**Business question:**  
What types of billing exceptions are occurring, how severe are they, and how many remain unresolved?

**Primary data source:**

- `Billing Exception Summary`

**KPI cards:**

- Total Exceptions
- Open Exceptions
- In Review Exceptions
- Resolved Exceptions
- High Severity Exceptions
- Affected Accounts

**Visuals:**

| Visual | Fields |
|---|---|
| Donut chart | `exception_type` by `exception_count` |
| Stacked bar chart | `exception_type` by `resolution_status` |
| Bar chart | `severity` by `exception_count` |
| Line chart | `billing_cycle_id` vs `exception_count` |
| Matrix | `exception_type`, `severity`, `resolution_status`, `exception_count`, `affected_accounts` |
| Bar chart | `county` by `exception_count` |

**Slicers:**

- Billing Cycle
- Exception Type
- Severity
- Resolution Status
- Customer Segment
- County

### Page 4: Service Request Backlog

**Business question:**  
What operational follow-up work is open, in progress, closed, or cancelled, and where is the backlog concentrated?

**Primary data source:**

- `Service Request Backlog`

**KPI cards:**

- Total Service Requests
- Open Service Requests
- In Progress Service Requests
- Closed Service Requests
- Critical Requests
- High Priority Requests

**Visuals:**

| Visual | Fields |
|---|---|
| Stacked bar chart | `request_type` by `request_status` and `service_request_count` |
| Bar chart | `priority` by `service_request_count` |
| Bar chart | `county` by `open_requests` |
| Matrix | `request_type`, `priority`, `request_status`, `service_request_count`, `affected_accounts` |
| Line chart | `billing_cycle_id` vs `service_request_count` |
| Donut chart | `request_status` by `service_request_count` |

**Slicers:**

- Billing Cycle
- Request Type
- Request Status
- Priority
- Customer Segme

### Page 5: Account Readiness

**Business question:**  
Which accounts are ready for normal billing/reporting or migration, and what issues are blocking readiness?

**Primary data source:**

- `Account Readiness`

**KPI cards:**

- Ready Accounts
- Not Ready Accounts
- Account Readiness Rate %
- Accounts with Missing Meter Reads
- Accounts with Invalid Meter Reads
- Accounts with Open Service Requests

**Visuals:**

| Visual | Fields |
|---|---|
| Donut chart | `readiness_status` by account count |
| Bar chart | `readiness_status` by account count |
| Bar chart | `county` by not-ready account count |
| Bar chart | `customer_segment` by readiness rate |
| Matrix | `account_id`, `customer_segment`, `county`, `account_status`, `meter_status`, `readiness_status` |
| Table | Accounts with missing reads, invalid reads, open exceptions, or open service requests |

**Slicers:**

- Readiness Status
- Readiness Flag
- Customer Segment
- County
- Account Status
- Meter Status

### Page 6: PG&E Outage Context

**Business question:**  
Where is current PG&E outage impact concentrated, and how does outage impact compare with county electricity consumption?

**Primary data sources:**

- `Outage Consumption Context`
- `Outage Area County Summary`
- `Outage Area Properties`
- Optional: `pge_outage_areas_clean.geojson`

**KPI cards:**

- Total Impacted Customers
- Total Outage Incidents
- Total Outage Areas
- Planned Outage Areas
- Not-Planned Outage Areas
- Average Estimated Restoration Hours

**Visuals:**

| Visual | Fields |
|---|---|
| Bar chart | `county` by `total_impacted_customers` |
| Bar chart | `county` by `outage_incidents` or `outage_area_count` |
| Scatter plot | `annual_total_gwh` vs `total_impacted_customers` by county |
| Bar chart | `county` by `impacted_customers_per_annual_gwh` |
| Table | County outage and consumption context |
| Map visual | County or GeoJSON outage area layer, if supported cleanly in Power BI |

**Slicers:**

- County
- Outage Type
- Planned vs Not Planned
- Utility Company
- Outage Status

**Design note:**

The first Power BI version can use county-level outage context tables for reliable mapping and filtering. The cleaned GeoJSON file can be added later as an optional polygon layer if Power BI handles the file cleanly through the selected map visual.

### Dashboard Page Design Summary

The dashboard is designed as a six-page operational analytics report:

1. **Executive Overview**: high-level Meter-to-Cash performance.
2. **Billing Performance**: bill status, amount, usage, and segment performance.
3. **Billing Exceptions**: exception type, severity, status, and affected accounts.
4. **Service Request Backlog**: operational follow-up work and backlog monitoring.
5. **Account Readiness**: data quality and migration-readiness blockers.
6. **PG&E Outage Context**: real outage impact and county electricity consumption context.

This structure creates a clear story from executive KPIs to operational detail, while connecting synthetic Meter-to-Cash workflows back to real PG&E outage and electricity consumption data.

### Dashboard Page Design Summary

The dashboard is designed as a six-page operational analytics report:

1. **Executive Overview**: high-level Meter-to-Cash performance.
2. **Billing Performance**: bill status, amount, usage, and segment performance.
3. **Billing Exceptions**: exception type, severity, status, and affected accounts.
4. **Service Request Backlog**: operational follow-up work and backlog monitoring.
5. **Account Readiness**: data quality and migration-readiness blockers.
6. **PG&E Outage Context**: real outage impact and county electricity consumption context.

This structure creates a clear story from executive KPIs to operational detail, while connecting synthetic Meter-to-Cash workflows back to real PG&E outage and electricity consumption data.

### Build Steps

1. **Create Power BI-ready workbook**
   - Run the final export section of this notebook.
   - Confirm that `data/powerbi/pge_powerbi_tables.xlsx` was created.

2. **Open Power BI Desktop**
   - Create a new report file.
   - Save it as `pge_meter_to_cash_dashboard.pbix`.

3. **Load the Power BI workbook**
   - Use **Get Data → Excel workbook**.
   - Load `data/powerbi/pge_powerbi_tables.xlsx`.
   - Select the relevant sheets:
     - `Monthly Billing Trend`
     - `Monthly KPIs`
     - `Bill Status Summary`
     - `Billing Exceptions`
     - `Service Requests`
     - `Account Readiness`
     - `Outage Context`
     - `Outage Area County`
     - `Outage Area Detail`

4. **Rename tables if needed**
   - Use readable names such as:
     - `Monthly Billing Trend`
     - `Monthly KPIs`
     - `Bill Status Summary`
     - `Billing Exception Summary`
     - `Service Request Backlog`
     - `Account Readiness`
     - `Outage Consumption Context`

5. **Check data types**
   - Dates should load as Date or Date/Time.
   - Amount, usage, count, and percentage fields should load as numeric.
   - IDs and categorical fields should load as text.

6. **Create DAX measures**
   - Add the core measures defined in this notebook.
   - Validate totals against the dashboard validation targets.

7. **Build dashboard pages**
   - Start with the Executive Overview page.
   - Then build Billing Performance, Billing Exceptions, Service Request Backlog, Account Readiness, and PG&E Outage Context.

### Dashboard Validation Targets

Use these expected totals to validate the Power BI report after loading the data:

| Metric | Expected Value |
|---|---:|
| Total Bills | 60,000 |
| Bills per Month | 5,000 |
| Total Accounts | 5,000 |
| Total Meter Reads | 60,000 |
| Billing Exceptions | 5,403 |
| Service Requests | 3,000 |
| UAT Test Cases | 10 |
| Account Readiness Records | 5,000 |
| Reporting Views Exported | 7 |
| Geospatial Output Files | 4 |

These values should match the outputs from Notebooks 2, 3, and 4. If Power BI totals differ, the most likely cause is a relationship or duplicate-counting issue.

## 7. Notebook Summary and Next Steps

This notebook prepared the Power BI dashboard design for the PG&E-style Utility Operations & Meter-to-Cash Analytics project.

Key outcomes:

- Identified the reporting-ready CSV files created from SQL views in Notebook 3.
- Identified the geospatial output files created in Notebook 4.
- Defined the planned Power BI dashboard pages.
- Documented the recommended Power BI tables to load.
- Defined the recommended data model and relationship approach.
- Listed core executive KPIs and DAX measures.
- Specified visuals, slicers, and business questions for each dashboard page.
- Created a recommended Power BI build order.
- Documented dashboard validation targets to prevent duplicate-counting and relationship issues.

The next step is to build the Power BI report using the cleaned Power BI workbook in data/powerbi. The workbook is created from the reporting files in data/reporting and the geospatial support files in data/geospatial.

The final dashboard should include six pages:

1. Executive Overview
2. Billing Performance
3. Billing Exceptions
4. Service Request Backlog
5. Account Readiness
6. PG&E Outage Context

After the Power BI dashboard is built, the remaining project work will be final documentation, including the README, data dictionary, business requirements summary, technical design summary, and screenshots for GitHub.

## 8. Create Power BI-Ready Excel Workbook

Power BI can interpret CSV numeric fields differently depending on regional settings, especially when the computer expects commas as decimal separators. To avoid type-conversion issues, this section creates a Power BI-ready Excel workbook with one sheet per reporting table.

The workbook will be saved to `data/powerbi` and used as the main Power BI input source instead of loading individual CSV files from `data/reporting`.

This creates a cleaner handoff layer:

- `data/reporting`: SQL view CSV outputs
- `data/geospatial`: geospatial outputs
- `data/powerbi`: final Power BI-ready workbook

In [5]:
%pip install XlsxWriter

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: C:\Users\Chris\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Define paths
REPORTING_DIR = Path("../data/reporting")
GEOSPATIAL_DIR = Path("../data/geospatial")
POWERBI_DIR = Path("../data/powerbi")

POWERBI_DIR.mkdir(parents=True, exist_ok=True)

POWERBI_WORKBOOK = POWERBI_DIR / "pge_powerbi_tables.xlsx"

# Delete old workbook if it exists
if POWERBI_WORKBOOK.exists():
    POWERBI_WORKBOOK.unlink()

# Excel does not like certain hidden XML/control characters
ILLEGAL_XML_RE = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")

def clean_excel_text(value):
    if isinstance(value, str):
        return ILLEGAL_XML_RE.sub("", value)
    return value

powerbi_sources = {
    "Monthly Billing Trend": REPORTING_DIR / "vw_monthly_billing_trend.csv",
    "Monthly KPIs": REPORTING_DIR / "vw_meter_to_cash_monthly_kpis.csv",
    "Bill Status Summary": REPORTING_DIR / "vw_bill_status_summary.csv",
    "Billing Exceptions": REPORTING_DIR / "vw_billing_exception_summary.csv",
    "Service Requests": REPORTING_DIR / "vw_service_request_backlog.csv",
    "Account Readiness": REPORTING_DIR / "vw_account_readiness.csv",
    "Outage Context": REPORTING_DIR / "vw_pge_outage_consumption_context.csv",
    "Outage Area County": GEOSPATIAL_DIR / "pge_outage_area_county_summary.csv",
    "Outage Area Detail": GEOSPATIAL_DIR / "pge_outage_area_properties_clean.csv"
}

date_columns = {
    "Monthly Billing Trend": ["cycle_start_date", "cycle_end_date"],
    "Outage Area Detail": ["start_datetime", "estimated_restoration_datetime"]
}

numeric_columns = {
    "Monthly Billing Trend": [
        "total_bills", "accounts_billed", "total_bill_amount", "avg_bill_amount",
        "total_kwh_usage", "paid_bills", "generated_bills", "past_due_bills",
        "exception_bills", "exception_rate_pct", "past_due_rate_pct"
    ],
    "Monthly KPIs": [
        "accounts_billed", "total_bills", "paid_bills", "generated_bills",
        "past_due_bills", "exception_bills", "total_bill_amount", "avg_bill_amount",
        "total_kwh_usage", "billing_exceptions", "open_exceptions",
        "in_review_exceptions", "resolved_exceptions", "service_requests",
        "open_service_requests", "in_progress_service_requests",
        "closed_service_requests", "bill_exception_rate_pct", "past_due_rate_pct"
    ],
    "Bill Status Summary": [
        "bill_count", "account_count", "total_bill_amount", "avg_bill_amount",
        "total_kwh_usage", "avg_kwh_usage", "paid_bills", "generated_bills",
        "past_due_bills", "exception_bills"
    ],
    "Billing Exceptions": [
        "exception_count", "affected_accounts", "open_exceptions",
        "in_review_exceptions", "resolved_exceptions",
        "high_severity_exceptions", "medium_severity_exceptions"
    ],
    "Service Requests": [
        "service_request_count", "affected_accounts", "open_requests",
        "in_progress_requests", "closed_requests", "cancelled_requests",
        "critical_requests", "high_priority_requests", "medium_priority_requests",
        "low_priority_requests"
    ],
    "Account Readiness": [
        "total_meter_reads", "missing_meter_reads", "invalid_meter_reads",
        "total_bills", "exception_bills", "past_due_bills",
        "total_billing_exceptions", "open_billing_exceptions",
        "in_review_billing_exceptions", "total_service_requests",
        "open_service_requests", "in_progress_service_requests",
        "readiness_flag"
    ],
    "Outage Context": [
        "outage_incidents", "total_impacted_customers", "avg_impacted_customers",
        "median_impacted_customers", "avg_estimated_restoration_hours",
        "max_estimated_restoration_hours", "annual_total_gwh",
        "annual_residential_gwh", "annual_nonresidential_gwh",
        "avg_monthly_gwh", "impacted_customers_per_annual_gwh",
        "incidents_per_annual_gwh"
    ],
    "Outage Area County": [
        "outage_area_count", "total_impacted_customers", "avg_impacted_customers",
        "median_impacted_customers", "planned_outage_areas",
        "not_planned_outage_areas", "avg_estimated_restoration_hours",
        "max_estimated_restoration_hours"
    ],
    "Outage Area Detail": [
        "object_id", "impacted_customers", "estimated_restoration_hours"
    ]
}

cleaned_tables = {}

for sheet_name, file_path in powerbi_sources.items():
    df = pd.read_csv(file_path)

    # Clean hidden invalid Excel/XML characters from all text columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(clean_excel_text)

    # Convert date columns to clean text dates
    for col in date_columns.get(sheet_name, []):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.strftime("%Y-%m-%d")

    # Convert numeric columns
    for col in numeric_columns.get(sheet_name, []):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Replace Excel-problem values
    df = df.replace([np.inf, -np.inf], np.nan)

    cleaned_tables[sheet_name] = df

# Use xlsxwriter instead of openpyxl
with pd.ExcelWriter(
    POWERBI_WORKBOOK,
    engine="xlsxwriter",
    engine_kwargs={"options": {"strings_to_urls": False, "nan_inf_to_errors": True}}
) as writer:
    for sheet_name, df in cleaned_tables.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Power BI workbook created:")
print(POWERBI_WORKBOOK)

print("\nSheets exported:")
for sheet_name, df in cleaned_tables.items():
    print(f"{sheet_name}: {df.shape[0]} rows, {df.shape[1]} columns")

Power BI workbook created:
..\data\powerbi\pge_powerbi_tables.xlsx

Sheets exported:
Monthly Billing Trend: 12 rows, 14 columns
Monthly KPIs: 12 rows, 20 columns
Bill Status Summary: 4870 rows, 15 columns
Billing Exceptions: 2314 rows, 13 columns
Service Requests: 2440 rows, 16 columns
Account Readiness: 5000 rows, 24 columns
Outage Context: 37 rows, 13 columns
Outage Area County: 33 rows, 9 columns
Outage Area Detail: 173 rows, 11 columns
